<a href="https://colab.research.google.com/github/mjh5153/pgp-generative-ai/blob/main/LangGraph_Course_Work.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
pip install -q langchain langchain-community langchain-core langchain-google-genai faiss-cpu python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.7/81.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 20.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1. State definition using TypedDict
class State(TypedDict):
    number: int
    result: int

# 2. Node definition
def square_node(state: State) -> dict:
    """
    Node Logic
    - Read the number from the State
    - Square it
    - Return the dictionary update for the state
    """
    num = state["number"]
    return {"result": num * num}

# 3. Build graph
graph = StateGraph(State)

# 4. Add node
graph.add_node("square", square_node)

# 5. Define the entry point
graph.add_edge(START, "square")

# 6. Edge (where to go next)
graph.add_edge("square", END)

# 7. Compile graph
app = graph.compile()

# 8. Run the graph
output = app.invoke({"number": 25})

print("Result: ", output["result"])

Result:  625


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# Memory management
# temporary - global - session
#                 - threads -> daemon threads<= implicitly
#                                        -> os -> Hy???

class State(TypedDict):
  number: int
  total: int # memory to keep updating one by one


def add_node(state: State) -> dict:
  """
    Reads:
      number (input)
      total (memory)
    Updates:
      total (memory)
    Returns:
      dictionary updating the state
  """
  current_total = state.get("total", 0) if state.get("total") is not None else 0
  return {
      "total": current_total + state["number"]
  }

# 3. Build graph
graph = StateGraph(State)

graph.add_node("add", add_node)
graph.add_edge(START, "add")
graph.add_edge("add", END)

app = graph.compile()

# 4. Use memory define mem mgmt
memory = {
  "total": 0
}

# First Run
run_1_output = app.invoke({"total": memory["total"], "number": 5})
memory.update(run_1_output)
print("Run-1: After updating the memory to 5, memory = ", memory["total"])

# Second Run
run_2_output = app.invoke({"total": memory["total"], "number": 10})
memory.update(run_2_output)
print("Run-2: After updating the memory to 10, memory = ", memory["total"])

Run-1: After updating the memory to 5, memory =  5
Run-2: After updating the memory to 10, memory =  15


In [ ]:
# Mock LLM Demo
from langgraph.graph import StateGraph, MessagesState, START, END

def mock_llm(state: MessagesState):
  return { "messages": [{
      "role": "ai",
      "content": "This is a mock response"
  }]}

graph = StateGraph(MessagesState)
graph.add_node("mock", mock_llm)
graph.add_edge(START, "mock")
graph.add_edge("mock", END)

app = graph.compile()

output = app.invoke({"messages": [
    {"role": "user",
     "content": "hi"
     }
]})
print(output["messages"])

[HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='9b351ef3-4bdf-4139-9653-05ea84e80c21'), AIMessage(content='This is a mock response', additional_kwargs={}, response_metadata={}, id='822c796a-054d-4102-abbd-624f0488843f', tool_calls=[], invalid_tool_calls=[])]


In [6]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, START, END

# Define the shared state between agents
class GlobalState(TypedDict):
    goal: str
    plan: Optional[str]
    result: Optional[str]

# 1. Planning Agent
def planner_agent(state: GlobalState) -> dict:
    goal = state["goal"]
    print(f"[Planner] Creating a plan for goal: '{goal}'")
    # Generate a step-by-step mock plan
    plan = f"1. Gather ingredients for {goal}. 2. Execute preparation steps. 3. Serve and enjoy!"
    return {"plan": plan}

# 2. Executor Agent
def executor_agent(state: GlobalState) -> dict:
    plan = state.get("plan")
    print(f"[Executor] Executing plan:\n{plan}")
    # Generate mock execution results based on the plan
    result = f"Successfully completed the following steps: {plan}"
    return {"result": result}

# 3. Build the workflow graph
workflow = StateGraph(GlobalState)

# 4. Add nodes
workflow.add_node("planner", planner_agent)
workflow.add_node("executor", executor_agent)

# 5. Define connections (edges)
workflow.add_edge(START, "planner")
workflow.add_edge("planner", "executor")
workflow.add_edge("executor", END)

# Compile the workflow
app = workflow.compile()

# Run the multi-agent graph
final_state = app.invoke({"goal": "make a cup of tea"})
print("\n--- Final Result ---")
print(final_state["result"])

[Planner] Creating a plan for goal: 'make a cup of tea'
[Executor] Executing plan:
1. Gather ingredients for make a cup of tea. 2. Execute preparation steps. 3. Serve and enjoy!

--- Final Result ---
Successfully completed the following steps: 1. Gather ingredients for make a cup of tea. 2. Execute preparation steps. 3. Serve and enjoy!


In [ ]:
# Agentic AI(LangGraph) with Agentic RAG(LangChain)
# Architecture, swe: Start with one Tool i.e. LangGraph/LangChain

In [15]:
# import statements
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph, END
from typing import TypedDict, Optional
from google.colab import userdata

import os

# =======================LangChain Coding=======================
# load the API key from Colab Userdata or fallback to .env / environment
try:
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    load_dotenv()
    GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# load the document ==> text document (data.txt)
loader = TextLoader(file_path="data.txt", encoding="utf-8")
documents = loader.load()

# Text Splitters
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

# Embeddings ==> using google's gemini-embedding-001 with explicit API key
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GOOGLE_API_KEY)

# Vector Store
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

# LLM model - Updated to gemini-2.5-flash
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", google_api_key=GOOGLE_API_KEY)

# prompt
prompt = ChatPromptTemplate.from_template(
    "Answer the question based only on the context:\n\n{context}\n\nQuestion: {question}"
)

# =======================LangGraph Coding=======================
# State Defn
class GraphState(TypedDict):
    question: str
    context: str
    answer: str

# nodes
def retrieve(state: GraphState):
    docs = retriever.invoke(state["question"])
    context = "\n\n".join(doc.page_content for doc in docs)
    return {
        "context": context
    }

def generate(state: GraphState):
    # LCEL
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke(
        {
            "context": state["context"],
            "question": state["question"]
        }
    )
    return {
        "answer": answer
    }

# build the graph
graph = StateGraph(GraphState)

# add the nodes
graph.add_node("retrieve", retrieve)

# add generate node
graph.add_node("generate", generate)

# workflow/edges
graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)

# compile the graph
app = graph.compile()

# run the query
query = "What is this document all about?"

res = app.invoke(
    {
        "question": query
    }
)

print(res["answer"])

Based on the provided context, the document is about the conception and origins of Lewis Carroll's *Alice's Adventures in Wonderland* (originally told as "Alice's Adventures Under Ground"). Specifically, it details the 4 July 1862 rowing trip with the Liddell sisters where Carroll first told the story to Alice Liddell, the writing of the manuscript, and scholarly discussions regarding the event (the "golden afternoon") and how much the character of Alice was based on Alice Liddell.
